# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema found at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Install mlcroissant if necessary
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their field `@id`s.
The schema may contain multiple record sets, each potentially with a unique structure.

Let's look at all the record set `@id`s, field `@id`s, and columns defined in the dataset.

In [ ]:
# Extract record set entities and their field information by @id
def print_record_sets_and_fields(metadata):
    record_sets = metadata.record_sets
    print("Record sets in the dataset:")
    rs_ids = []
    for rs in record_sets:
        print(f"  - RecordSet: {rs['@id']}")
        rs_ids.append(rs['@id'])
        print(f"    Name: {rs.get('name','')}")
        print(f"    Description: {rs.get('description','')}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print(f"    Fields:")
        for f in fields:
            print(f"      - Field @id: {f['@id']}")
            print(f"        Name: {f.get('name','')}")
            print(f"        Data type: {f.get('dataType', '')}")
    print()
    return rs_ids

record_set_ids = print_record_sets_and_fields(metadata)

## 3. Data Extraction
Let's extract data from each record set into a pandas DataFrame using their `@id`. You can use the `@id` values printed above to select a particular record set. For demonstration, we'll use the first available one.

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Head:")
        display(df.head())
    except Exception as e:
        print(f"  Could not load or display data for {record_set_id}: {e}")

# For demonstration, select the first record set
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"Using primary RecordSet for further analysis: {selected_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
We will perform some basic EDA:
- Filtering rows by a numeric field
- Normalizing the numeric column
- Grouping by a categorical field

Replace `<numeric_field_id>` and `<group_field_id>` below with the actual field (column) names derived from the previous output.

In [ ]:
# Helper function to guess numeric and group fields from the DataFrame
def guess_numeric_and_group_fields(df):
    numeric_field = None
    group_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            if numeric_field is None:
                numeric_field = col
        else:
            if group_field is None:
                group_field = col
    return numeric_field, group_field

if record_set_ids:
    df = dataframes[selected_record_set_id]
    numeric_field_id, group_field_id = guess_numeric_and_group_fields(df)
    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")

    # Filter numeric_field_id > threshold (e.g., use median or 10)
    threshold = df[numeric_field_id].median() if numeric_field_id else 0
    filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id else df.copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    if numeric_field_id:
        filtered_df[numeric_field_id + '_normalized'] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by group_field_id if present
    if group_field_id and group_field_id in filtered_df.columns and numeric_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and optionally, group means.

If you have specific fields of interest, replace the field names accordingly.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We explored the FAIR² dataset using `mlcroissant`, loaded available record sets, and performed simple EDA and visualization. This notebook can be extended for deeper analysis based on the research question or field selection.